# 04 — Q-Learning and the SARSA vs Q-Learning Showdown
**Week 5 | Model-Free Learning**

Q-Learning uses the **greedy next action** in its update — regardless of what the agent actually does:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ R + \gamma \; \underbrace{\max_{a'} Q(s', a')}_{\text{greedy, not taken}} - Q(s,a) \Big]$$

**One-line difference from SARSA:**
- SARSA: `Q[s,a] += alpha * (r + gamma * Q[ns, next_action_taken] - Q[s,a])`
- Q-Learning: `Q[s,a] += alpha * (r + gamma * max(Q[ns]) - Q[s,a])`

This makes Q-Learning **off-policy** — it learns the optimal policy even while exploring.

In [ ]:
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable,'-m','pip','install','gymnasium','-q']); import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

env = gym.make('Taxi-v3')
n_s = env.observation_space.n; n_a = env.action_space.n

## 1. Q-Learning Implementation

In [ ]:
def q_learning(env, n_episodes=15_000, alpha=0.1, gamma=0.99,
               eps_start=1.0, eps_end=0.01, eps_decay=0.999):
    Q = np.zeros((n_s, n_a))
    eps = eps_start
    returns_per_ep = []
    steps_per_ep = []

    for ep in range(n_episodes):
        s, _ = env.reset()
        ep_return = 0.0; steps = 0; done = False
        while not done:
            # Choose action (ε-greedy behaviour policy)
            a = env.action_space.sample() if np.random.rand()<eps else np.argmax(Q[s])
            ns, r, term, trunc, _ = env.step(a)
            done = term or trunc

            # Q-Learning update — uses max Q[ns], NOT the action taken
            Q[s,a] += alpha * (r + gamma * np.max(Q[ns]) * (not done) - Q[s,a])

            s = ns; ep_return += r; steps += 1

        eps = max(eps_end, eps * eps_decay)
        returns_per_ep.append(ep_return)
        steps_per_ep.append(steps)

    return Q, returns_per_ep, steps_per_ep

print("Training Q-Learning on Taxi-v3...")
Q_ql, returns_ql, steps_ql = q_learning(env)
print("Done.")

## 2. SARSA vs Q-Learning — Direct Comparison
Train both with the same hyperparameters and seed.

In [ ]:
def sarsa(env, n_episodes=15_000, alpha=0.1, gamma=0.99, eps_start=1.0, eps_end=0.01, eps_decay=0.999):
    Q = np.zeros((n_s, n_a)); eps = eps_start
    returns_per_ep = []
    for ep in range(n_episodes):
        s,_ = env.reset()
        a = env.action_space.sample() if np.random.rand()<eps else np.argmax(Q[s])
        ep_r = 0; done = False
        while not done:
            ns,r,term,trunc,_ = env.step(a)
            done=term or trunc
            na = env.action_space.sample() if np.random.rand()<eps else np.argmax(Q[ns])
            Q[s,a] += alpha*(r + gamma*Q[ns,na]*(not done) - Q[s,a])
            s,a = ns,na; ep_r+=r
        eps=max(eps_end, eps*eps_decay)
        returns_per_ep.append(ep_r)
    return Q, returns_per_ep

np.random.seed(42)
Q_s, ret_s = sarsa(env)
np.random.seed(42)
Q_q, ret_q = q_learning(env)[:2]

window = 300
fig, ax = plt.subplots(figsize=(10, 4))
for ret, label, color in [(ret_s,'SARSA','steelblue'), (ret_q,'Q-Learning','tomato')]:
    rolling = np.convolve(ret, np.ones(window)/window, mode='valid')
    ax.plot(rolling, color=color, linewidth=1.8, label=label)
ax.set_xlabel('Episode'); ax.set_ylabel(f'Avg Return (rolling {window})')
ax.set_title('SARSA vs Q-Learning on Taxi-v3')
ax.legend(fontsize=11); plt.tight_layout(); plt.show()

## 3. Evaluate and Compare Final Policies

In [ ]:
def evaluate(env, Q, n_episodes=500):
    rewards = []
    for _ in range(n_episodes):
        s,_=env.reset(); ep_r=0; done=False
        for _ in range(200):
            s,r,term,trunc,_=env.step(np.argmax(Q[s]))
            ep_r+=r; done=term or trunc
            if done: break
        rewards.append(ep_r)
    return np.mean(rewards), np.std(rewards)

for label, Q in [('SARSA', Q_s), ('Q-Learning', Q_q)]:
    mean_r, std_r = evaluate(env, Q)
    print(f"{label:12}: mean={mean_r:6.2f} ± {std_r:.2f}")

## 4. Q-Table Heatmap

In [ ]:
action_names = ['South','North','East','West','Pickup','Dropoff']
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, Q, title in [(axes[0],Q_s,'SARSA'),(axes[1],Q_q,'Q-Learning')]:
    im = ax.imshow(Q[:50,:], cmap='RdYlGn', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(n_a)); ax.set_xticklabels(action_names, rotation=20, ha='right')
    ax.set_ylabel('State (first 50)'); ax.set_title(f'Q-Table — {title}')
plt.tight_layout(); plt.show()

## 5. Key Conceptual Difference: The Cliff Walk
SARSA learns a SAFER route; Q-Learning learns the OPTIMAL but riskier route.

In [ ]:
try:
    env_cliff = gym.make('CliffWalking-v0')
    ns_c = env_cliff.observation_space.n; na_c = env_cliff.action_space.n
    def q_learning_cliff(env, n_ep=5000, alpha=0.5, gamma=1.0, eps=0.1):
        Q=np.zeros((ns_c,na_c)); rets=[]
        for _ in range(n_ep):
            s,_=env.reset(); ep_r=0; done=False
            while not done:
                a=env.action_space.sample() if np.random.rand()<eps else np.argmax(Q[s])
                ns,r,term,trunc,_=env.step(a); done=term or trunc
                Q[s,a]+=alpha*(r+gamma*np.max(Q[ns])*(not done)-Q[s,a]); s=ns; ep_r+=r
            rets.append(ep_r)
        return Q,rets
    def sarsa_cliff(env, n_ep=5000, alpha=0.5, gamma=1.0, eps=0.1):
        Q=np.zeros((ns_c,na_c)); rets=[]
        for _ in range(n_ep):
            s,_=env.reset()
            a=env.action_space.sample() if np.random.rand()<eps else np.argmax(Q[s])
            ep_r=0; done=False
            while not done:
                ns,r,term,trunc,_=env.step(a); done=term or trunc
                na=env.action_space.sample() if np.random.rand()<eps else np.argmax(Q[ns])
                Q[s,a]+=alpha*(r+gamma*Q[ns,na]*(not done)-Q[s,a]); s,a=ns,na; ep_r+=r
            rets.append(ep_r)
        return Q,rets
    _,ret_ql_c = q_learning_cliff(env_cliff)
    _,ret_s_c  = sarsa_cliff(env_cliff)
    w=100; fig,ax=plt.subplots(figsize=(9,3.5))
    ax.plot(np.convolve(ret_ql_c,np.ones(w)/w,'valid'),color='tomato',label='Q-Learning',lw=1.8)
    ax.plot(np.convolve(ret_s_c, np.ones(w)/w,'valid'),color='steelblue',label='SARSA',lw=1.8)
    ax.set_xlabel('Episode'); ax.set_ylabel('Return'); ax.set_title('CliffWalking: SARSA is safer, Q-Learning is optimal')
    ax.legend(); plt.tight_layout(); plt.show()
    print("SARSA stays away from cliff (safe). Q-Learning walks the cliff edge (optimal but risky during training).")
except Exception as e:
    print(f"CliffWalking env not available or error: {e}")

## ✅ Exercises
1. Set ε=0 from the start (greedy). Does Q-Learning still converge? What about SARSA?
2. Double α to 0.2. Does Q-Learning become unstable before SARSA does?
3. **Challenge**: implement **Double Q-Learning** (maintain two Q-tables Q1 and Q2, alternate updates). Does it reduce overestimation bias on Taxi-v3?

Ans 1)

In [ ]:
Q_q0,	returns_q0	=	q_learning(env,	n_episodes=15_000,	eps_start=0.0,	eps_end=0.0)
Q_s0g,	returns_s0g	=	None,	None
try:
				Q_s0g,	returns_s0g	=	sarsa(env,	n_episodes=15_000,	eps_start=0.0,	eps_end=0.0,	eps_decay=1.0)
except	TypeError:
				pass		#	if	your	sarsa()	signature	differs,	mirror	the	eps	args	accordingly
for	label,	Q	in	[('Q-learning	ε=0',	Q_q0)]	+	([('SARSA	ε=0',	Q_s0g)]	
if	Q_s0g	is	not	None	else	[]):
				m,	s_	=	evaluate(env,	Q)
				print(f"{label}:	greedy	return	=	{m:.2f}	±	{s_:.2f}")

Ans 2 - Neither	becomes	unstable	—	but	SARSA’s	curve	is
visibly	noisier,	and	the	premise	of	the	question	is	backwards	for
this	environment

In [ ]:
Q_q2,	returns_q2	=	q_learning(env,	n_episodes=15_000,	alpha=0.2)
Q_s2,	returns_s2,	_	=	sarsa(env,	n_episodes=15_000,	alpha=0.2)
window	=	300
plt.figure(figsize=(9,	3.5))
for	rets,	label,	color	in	[(returns_q2,	'Q-learning	α=0.2',	'seagreen'),
																											(returns_s2,	'SARSA	α=0.2',	'steelblue')]:
				plt.plot(np.convolve(rets,	np.ones(window)/window,	mode='valid'),
			label=label,	color=color,	linewidth=1.5)
plt.xlabel('Episode');	plt.ylabel('Avg	return')
plt.title('α=0.2:	Q-learning	vs	SARSA');	plt.legend()
plt.tight_layout();	plt.show()
for	label,	Q	in	[('Q-learning	α=0.2',	Q_q2),	('SARSA	α=0.2',	Q_s2)]:
				m,	s_	=	evaluate(env,	Q)
				print(f"{label}:	greedy	return	=	{m:.2f}	±	{s_:.2f}")


Ans 3)

In [ ]:
def	double_q_learning(env,	n_episodes=15_000,	alpha=0.1,	gamma=0.99,
																						eps_start=1.0,	eps_end=0.01,	eps_decay=0.999):
				Q1	=	np.zeros((n_s,	n_a));	Q2	=	np.zeros((n_s,	n_a))
				eps	=	eps_start;	returns_per_ep	=	[]
				for	ep	in	range(n_episodes):
								s,	_	=	env.reset();	ep_return	=	0.0;	done	=	False
								while	not	done:
												if	np.random.rand()	<	eps:	a	=	env.action_space.sample()
												else:																						a	=	np.argmax(Q1[s]	+	Q2[s])			#	act	on	the	sum
												ns,	r,	term,	trunc,	_	=	env.step(a);	done	=	term	or	trunc
												if	np.random.rand()	<	0.5:
																best	=	np.argmax(Q1[ns])																														
#	select	with	Q1...
																Q1[s,	a]	+=	alpha	*	(r	+	gamma	*	Q2[ns,	best]	*	(not	done)	-	Q1[s,	a])		#	...evaluate	with	Q2
												else:
																best	=	np.argmax(Q2[ns])
																Q2[s,	a]	+=	alpha	*	(r	+	gamma	*	Q1[ns,	best]	*	(not	done)	-	Q2[s,	a])
												s	=	ns;	ep_return	+=	r
								eps	=	max(eps_end,	eps	*	eps_decay)
								returns_per_ep.append(ep_return)
				return	Q1,	Q2,	returns_per_ep
Q1,	Q2,	returns_dq	=	double_q_learning(env)
Q_dq	=	(Q1	+	Q2)	/	2
m,	s_	=	evaluate(env,	Q_dq)
print(f"Double	Q-learning:	greedy	return	=	{m:.2f}	±	{s_:.2f}")
#	Overestimation	check:	compare	each	method's	predicted	start-state	value
#	to	what	its	greedy	policy	ACTUALLY	earns.
for	label,	Q	in	[('Q-learning',	Q_q),	('Double	Q',	Q_dq)]:
				actual,	_	=	evaluate(env,	Q,	n_episodes=1000)
				predicted	=	np.mean([Q[env.reset()[0]].max()	for	_	in	
range(200)])
				print(f"{label:12}:	predicted	V(start)	=	{predicted:6.2f} actual	return	=	{actual:6.2f}			gap	=	{predicted-actual:+.2f}")
